In [3]:
# =============================================================================
# NOTEBOOK: 02_agent2_time_estimation.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — A Domain-Agnostic Three-Agent Pipeline (DSRM Design Artifact)
#
# AGENT 2 of 3 — Work-Time Estimation (with uncertainty tagging)
#   Input : artifacts/inference/process_graph.json   (AS-IS + TO-BE nodes)
#           data/raw/interview_tcb.txt                (qualitative time cues)
#   Method: For each node, estimate minutes_per_case and cases_per_month using
#           (a) interview cues and (b) shape-type priors. Every estimate is
#           TAGGED with a source (stated | implied | prior) and a confidence.
#           Self-Consistency (Wang et al., 2023): sample N stochastic rollouts
#           (temperature > 0) and reduce to the MEDIAN; dispersion across
#           rollouts is recorded as the Reliability signal.
#   Output: artifacts/inference/agent2_time.json
#
# Design principles realized here:
#   DP2 (compute in code): monthly_minutes = minutes_per_case * cases_per_month
#        is NEVER produced by the LLM — Python computes it, blocking arithmetic
#        hallucination.
#   DP3 (mark the unknown): the interview contains NO explicit numbers, so each
#        estimate is transparently labeled 'prior' or 'implied' rather than
#        passed off as a stated fact. Clarifying questions are emitted for the
#        lowest-confidence, highest-impact nodes (Robustness axis).
#
# TO-BE human effort uses the per-node human_effort_factor set in nb 015:
#   full -> 0.0, partial -> PARTIAL_RETAIN (0.30), manual -> 1.0
# This yields Metric B (time-weighted human-effort reduction), the primary
# ROI input consumed by Agent 3.
#
# All example content and code are in English for journal submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Bootstrap foundation from Notebook 00 (self-contained)
# =============================================================================
import os
import re
import json
import time
import hashlib
import statistics
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_RAW  = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "artifacts"
INFER     = ARTIFACTS / "inference"
TAB       = ARTIFACTS / "tables"
CACHE     = ARTIFACTS / "llm_cache"
for p in (INFER, TAB, CACHE):
    p.mkdir(parents=True, exist_ok=True)


def rel(p) -> str:
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


SEED = 42
np.random.seed(SEED)

RUN_MANIFEST = ARTIFACTS / "run_manifest.json"
if not RUN_MANIFEST.exists():
    raise FileNotFoundError("[ERROR] run_manifest.json missing. Run nb 00 first.")
manifest = json.loads(RUN_MANIFEST.read_text(encoding="utf-8"))

MODELS       = manifest["models"]
DEFAULT_TIER = manifest["default_tier"]
AUTO_GRADES  = manifest["auto_grades"]
MISSING      = manifest["missing_sentinel"]

load_dotenv(ROOT / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env.")
client = OpenAI(api_key=OPENAI_API_KEY)


# %%
# =============================================================================
# Cell 2. Re-declare cost tracker + llm_call() (identical to nb 00)
# =============================================================================
class CostTracker:
    def __init__(self):
        self.records: list[dict] = []

    def add(self, tier, model, usage, tag=""):
        p = MODELS.get(tier, {})
        pin, pcached, pout = p.get("in"), p.get("cached_in"), p.get("out")
        pt = getattr(usage, "prompt_tokens", 0) or 0
        ct = getattr(usage, "completion_tokens", 0) or 0
        cached = 0
        det = getattr(usage, "prompt_tokens_details", None)
        if det is not None:
            cached = getattr(det, "cached_tokens", 0) or 0
        fresh = max(pt - cached, 0)
        cost = None
        if None not in (pin, pout):
            pc = pcached if pcached is not None else pin
            cost = (fresh * pin + cached * pc + ct * pout) / 1_000_000
        self.records.append({"tag": tag, "tier": tier, "model": model,
                             "prompt_tokens": pt, "cached_tokens": cached,
                             "completion_tokens": ct, "cost_usd": cost})
        return cost or 0.0

    def total_usd(self):
        return float(sum(r["cost_usd"] or 0.0 for r in self.records))


COST = CostTracker()


def _safe_json(text):
    if not text:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    for opener, closer in (("{", "}"), ("[", "]")):
        i, j = t.find(opener), t.rfind(closer)
        if 0 <= i < j:
            try:
                return json.loads(t[i:j + 1])
            except Exception:
                continue
    return None


def _cache_key(model, system, user, temperature, response_json, salt=""):
    raw = json.dumps({"m": model, "s": system, "u": user, "t": temperature,
                      "j": response_json, "salt": salt},
                     ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def llm_call(system, user, tier=DEFAULT_TIER, temperature=0.0,
             response_json=True, tag="", use_cache=True, max_retries=3, salt=""):
    """Same as nb 00, plus a 'salt' so repeated Self-Consistency rollouts get
    distinct cache keys instead of colliding on one cached answer."""
    model = MODELS[tier]["name"]
    key = _cache_key(model, system, user, temperature, response_json, salt)
    cache_file = CACHE / f"{key}.json"
    if use_cache and cache_file.exists():
        c = json.loads(cache_file.read_text(encoding="utf-8"))
        c["cached"] = True
        # Preserve the ORIGINAL (already-paid) cost so the ledger reflects the
        # true analysis cost even on cached re-runs; also re-log it to COST so
        # cumulative spend is complete regardless of cache state.
        original_cost = c.get("cost_usd", 0.0) or 0.0
        COST.records.append({
            "tag": tag or c.get("model", ""), "tier": c.get("tier", tier),
            "model": c.get("model", ""), "prompt_tokens": 0,
            "cached_tokens": 0, "completion_tokens": 0,
            "cost_usd": original_cost,
        })
        c["cost_usd"] = original_cost
        return c
    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
    }
    if response_json:
        kwargs["response_format"] = {"type": "json_object"}
    kwargs["temperature"] = temperature
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            parsed = _safe_json(text) if response_json else None
            if response_json and parsed is None:
                raise ValueError("Response was not valid JSON.")
            cost = COST.add(tier, model, resp.usage, tag=tag or model)
            out = {"text": text, "json": parsed, "cached": False,
                   "tier": tier, "model": model, "cost_usd": cost}
            if use_cache:
                cache_file.write_text(json.dumps(out, ensure_ascii=False),
                                      encoding="utf-8")
            return out
        except TypeError as e:
            if "temperature" in kwargs:
                kwargs.pop("temperature", None); last_err = e; continue
            last_err = e
        except Exception as e:
            last_err = e; time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"[llm_call] failed after {max_retries} retries: {last_err}")


print("[INFO] CostTracker, llm_call() re-established for nb 02.")


# %%
# =============================================================================
# Cell 3. Load the process graph and the interview text
# =============================================================================
GRAPH_PATH = INFER / "process_graph.json"
if not GRAPH_PATH.exists():
    raise FileNotFoundError("[ERROR] process_graph.json missing. Run nb 015 first.")
G = json.loads(GRAPH_PATH.read_text(encoding="utf-8"))
asis_nodes = G["as_is"]["nodes"]
PARTIAL_RETAIN = G["params"]["partial_retain"]

INTERVIEW_PATH = DATA_RAW / "interview_tcb.txt"
INTERVIEW_TEXT = INTERVIEW_PATH.read_text(encoding="utf-8").strip()
print(f"[INFO] AS-IS nodes: {len(asis_nodes)}  | interview: {len(INTERVIEW_TEXT)} chars")
print(f"[INFO] partial_retain (TO-BE residual human effort) = {PARTIAL_RETAIN}")


# %%
# =============================================================================
# Cell 4. Estimation configuration — shape-type priors + qualitative weights
#
# These are EDITABLE priors, not measured constants. They encode a transparent
# default when the interview states no numbers. Every field is a parameter so a
# user can retune per engagement; nothing is domain-specific.
# =============================================================================
# Base minutes-per-case prior by flowchart shape type.
SHAPE_PRIOR_MIN = {
    "start": 3.0,     # intake / trigger — short
    "task": 15.0,     # ordinary activity
    "decision": 8.0,  # judgment / branch
    "end": 5.0,       # closeout / dispatch
}

# Global default monthly case volume (interview states none). Agent 3 will
# sweep this in sensitivity analysis, so a single transparent default is honest.
DEFAULT_CASES_PER_MONTH = 20.0

# Qualitative multipliers applied when the interview flags a task as unusually
# heavy or wasteful. Keys are neutral concepts; the LLM maps nodes to them.
QUALITATIVE_WEIGHT = {
    "most_time_consuming": 4.0,   # explicitly named as the biggest time sink
    "repeated_effort": 2.0,       # "a lot of repeated copy-paste / editing"
    "manual_data_entry": 2.0,     # "manual entry ... regrettable time sink"
    "normal": 1.0,                # default
}

# Self-Consistency settings (Reliability axis).
N_ROLLOUTS = manifest["pipeline_cfg"]["agent2_time"]["n_samples"]   # e.g. 5
SC_TEMPERATURE = manifest["pipeline_cfg"]["agent2_time"]["temperature"]  # e.g. 0.7

print(f"[INFO] Shape priors (min): {SHAPE_PRIOR_MIN}")
print(f"[INFO] Default cases/month: {DEFAULT_CASES_PER_MONTH}")
print(f"[INFO] Self-Consistency: N={N_ROLLOUTS} rollouts @ temp={SC_TEMPERATURE}")


# %%
# =============================================================================
# Cell 5. Agent-2 prompt — per-node time estimate with source tagging
#
# The LLM does NOT compute monthly totals (that is code, DP2). It only proposes,
# per node: minutes_per_case, cases_per_month, a qualitative weight key, a
# SOURCE tag, a confidence in [0,1], and a one-line rationale. The prompt is
# domain-agnostic; it reasons from the transcript plus the shape-type context.
# =============================================================================
node_ctx = [
    {"id": n["id"], "name": n["name"], "type": n["type"],
     "actor": n.get("lane", MISSING), "grade": n.get("grade", MISSING)}
    for n in asis_nodes
]

AGENT2_SYSTEM = f"""\
You are Agent 2 (Work-Time Estimation) in an ROI-estimation pipeline. You are
given an operational interview transcript (any domain) and a list of process
nodes already extracted from it. For EACH node, estimate how long the task takes
and how often it runs — but be explicit about how sure you are.

For each node output:
  - "minutes_per_case": a positive number — time for ONE occurrence of the task.
  - "cases_per_month": a positive number — how many times per month it runs.
  - "weight_key": one of {list(QUALITATIVE_WEIGHT)} — pick "most_time_consuming",
     "repeated_effort", or "manual_data_entry" ONLY if the transcript clearly
     signals it for this node; otherwise "normal".
  - "source": one of:
       "stated"  — the transcript gives an explicit number for this task,
       "implied" — no number, but the transcript qualitatively signals its size,
       "prior"   — neither; you are relying on a generic default for its type.
  - "confidence": a number in [0,1] reflecting how well-grounded the estimate is
     (stated ~0.9, implied ~0.5, prior ~0.3).
  - "rationale": one sentence citing the transcript phrase or the default basis.

Rules:
  - Do NOT compute monthly totals; only per-case time and monthly frequency.
  - Do NOT fabricate explicit numbers. If the transcript has none for a node,
    the source MUST be "implied" or "prior", never "stated".
  - Keep estimates realistic and internally consistent across similar tasks.

Return ONLY JSON:
{{ "estimates": [
     {{ "id": "wX", "minutes_per_case": <num>, "cases_per_month": <num>,
        "weight_key": "...", "source": "stated|implied|prior",
        "confidence": <0..1>, "rationale": "..." }}
] }}
"""

def build_user(nodes_ctx: list[dict]) -> str:
    return (
        "--- INTERVIEW TRANSCRIPT ---\n" + INTERVIEW_TEXT +
        "\n--- END TRANSCRIPT ---\n\nProcess nodes to estimate:\n" +
        json.dumps(nodes_ctx, ensure_ascii=False, indent=1)
    )

AGENT2_USER = build_user(node_ctx)
print(f"[INFO] Agent-2 prompt ready (system {len(AGENT2_SYSTEM)} chars, "
      f"user {len(AGENT2_USER)} chars).")


# %%
# =============================================================================
# Cell 6. Self-Consistency: run N stochastic rollouts, collect per-node samples
# =============================================================================
def run_rollouts(n: int) -> list[dict]:
    """Return a list of {id -> estimate dict} maps, one per rollout."""
    maps = []
    for k in range(n):
        r = llm_call(
            system=AGENT2_SYSTEM, user=AGENT2_USER,
            tier=DEFAULT_TIER, temperature=SC_TEMPERATURE,
            response_json=True, tag=f"agent2_rollout_{k}",
            use_cache=True, salt=f"rollout-{k}",   # distinct cache per rollout
        )
        est = (r["json"] or {}).get("estimates", [])
        m = {e["id"]: e for e in est if isinstance(e, dict) and e.get("id")}
        maps.append(m)
        print(f"   rollout {k+1}/{n}: {len(m)} estimates "
              f"(cached={r['cached']}, cost=${r['cost_usd']:.5f})")
    return maps


print(f"[INFO] Running {N_ROLLOUTS} Self-Consistency rollouts ...")
rollout_maps = run_rollouts(N_ROLLOUTS)


# %%
# =============================================================================
# Cell 7. Reduce rollouts -> median estimate + dispersion (Reliability signal)
#
# For each node we take the MEDIAN of minutes_per_case and cases_per_month
# across rollouts (robust to outliers), and record the coefficient of variation
# (CV = std/mean) as a per-node reliability signal. The most common source tag
# and the mean confidence are carried through.
# =============================================================================
def _nums(maps, nid, field):
    vals = []
    for m in maps:
        v = m.get(nid, {}).get(field)
        if isinstance(v, (int, float)) and v > 0:
            vals.append(float(v))
    return vals

def _cv(vals):
    if len(vals) < 2:
        return 0.0
    mu = statistics.mean(vals)
    return (statistics.pstdev(vals) / mu) if mu else 0.0

reduced = []
for n in asis_nodes:
    nid = n["id"]
    mpc_vals = _nums(rollout_maps, nid, "minutes_per_case")
    cpm_vals = _nums(rollout_maps, nid, "cases_per_month")

    # Fallbacks to shape-type prior / global default when a rollout omitted a node.
    mpc = statistics.median(mpc_vals) if mpc_vals else SHAPE_PRIOR_MIN.get(n["type"], 15.0)
    cpm = statistics.median(cpm_vals) if cpm_vals else DEFAULT_CASES_PER_MONTH

    # Source + confidence: majority source, mean confidence across rollouts.
    srcs = [rollout_maps[i].get(nid, {}).get("source", "prior")
            for i in range(len(rollout_maps)) if nid in rollout_maps[i]]
    source = max(set(srcs), key=srcs.count) if srcs else "prior"
    confs = [rollout_maps[i].get(nid, {}).get("confidence", 0.3)
             for i in range(len(rollout_maps)) if nid in rollout_maps[i]]
    confidence = round(float(statistics.mean(confs)), 3) if confs else 0.3
    wkeys = [rollout_maps[i].get(nid, {}).get("weight_key", "normal")
             for i in range(len(rollout_maps)) if nid in rollout_maps[i]]
    weight_key = max(set(wkeys), key=wkeys.count) if wkeys else "normal"

    reduced.append({
        "id": nid, "name": n["name"], "type": n["type"],
        "lane": n.get("lane", MISSING), "grade": n.get("grade", MISSING),
        "minutes_per_case": round(mpc, 2),
        "cases_per_month": round(cpm, 2),
        "weight_key": weight_key,
        "source": source, "confidence": confidence,
        "cv_minutes": round(_cv(mpc_vals), 3),   # reliability signal
        "n_samples": len(mpc_vals),
    })

print(f"[INFO] Reduced {len(reduced)} node estimates across "
      f"{N_ROLLOUTS} rollouts.")


# %%
# =============================================================================
# Cell 8. Compute effort IN CODE (DP2): AS-IS and TO-BE monthly minutes
#
# monthly_minutes = minutes_per_case * cases_per_month * qualitative_weight
# AS-IS human effort: nodes NOT already AI-performed contribute full effort.
# TO-BE human effort: multiply by the per-node human_effort_factor implied by
#   its grade (full->0, partial->PARTIAL_RETAIN, manual->1).
# =============================================================================
AI_LANES = {"system", "gpt", "ai", "ai agent"}

def is_ai(lane: str) -> bool:
    l = (lane or "").lower()
    return l in AI_LANES or "gpt" in l

def human_factor(grade: str) -> float:
    return {"full": 0.0, "partial": PARTIAL_RETAIN, "manual": 1.0}.get(grade, 1.0)

rows = []
asis_human_min = 0.0
tobe_human_min = 0.0
for e in reduced:
    w = QUALITATIVE_WEIGHT.get(e["weight_key"], 1.0)
    monthly = e["minutes_per_case"] * e["cases_per_month"] * w   # DP2: code, not LLM

    asis_is_human = not is_ai(e["lane"])
    asis_min = monthly if asis_is_human else 0.0
    # TO-BE: if AS-IS was human, apply the grade's human factor; AI stays 0.
    tobe_min = (monthly * human_factor(e["grade"])) if asis_is_human else 0.0

    asis_human_min += asis_min
    tobe_human_min += tobe_min
    rows.append({**e, "qual_weight": w,
                 "monthly_minutes": round(monthly, 1),
                 "asis_human_minutes": round(asis_min, 1),
                 "tobe_human_minutes": round(tobe_min, 1)})

df2 = pd.DataFrame(rows)
effort_reduction_pct = (1 - tobe_human_min / asis_human_min) * 100 if asis_human_min else 0.0

print(f"[INFO] AS-IS human effort : {asis_human_min:,.0f} min/month "
      f"({asis_human_min/60:,.1f} h)")
print(f"[INFO] TO-BE human effort : {tobe_human_min:,.0f} min/month "
      f"({tobe_human_min/60:,.1f} h)")
print(f"[Metric B] Human-effort reduction: {effort_reduction_pct:.1f}%  "
      f"<-- primary ROI input")


# %%
# =============================================================================
# Cell 9. Robustness: clarifying questions for low-confidence, high-impact nodes
#
# DP3 in action: rather than silently trusting weak estimates, flag the nodes
# where a human should confirm — those with low confidence AND large effort.
# These questions are an artifact of the Robustness axis, not a failure.
# =============================================================================
CONF_THRESHOLD = 0.4
impact = df2["asis_human_minutes"].fillna(0)
flagged = df2[(df2["confidence"] < CONF_THRESHOLD) & (impact > impact.median())]
clarifying = []
for _, r in flagged.iterrows():
    clarifying.append({
        "id": r["id"], "name": r["name"],
        "why": f"low confidence ({r['confidence']}) but high effort "
               f"({r['asis_human_minutes']:.0f} min/mo)",
        "question": f"For '{r['name']}', what is the actual time per case and "
                    f"the monthly volume? (current estimate is a "
                    f"{r['source']}-based guess)",
    })
print(f"[INFO] {len(clarifying)} clarifying question(s) generated "
      f"(low-confidence & high-impact nodes).")
for c in clarifying[:5]:
    print("   -", c["question"])


# %%
# =============================================================================
# Cell 10. Persist Agent-2 output + write Metric B back to the process graph
# =============================================================================
OUT_PATH = INFER / "agent2_time.json"
out_obj = {
    "agent": "agent2_time_estimation",
    "n_rollouts": N_ROLLOUTS,
    "sc_temperature": SC_TEMPERATURE,
    "params": {
        "shape_prior_min": SHAPE_PRIOR_MIN,
        "default_cases_per_month": DEFAULT_CASES_PER_MONTH,
        "qualitative_weight": QUALITATIVE_WEIGHT,
        "partial_retain": PARTIAL_RETAIN,
    },
    "totals": {
        "asis_human_minutes_per_month": round(asis_human_min, 1),
        "tobe_human_minutes_per_month": round(tobe_human_min, 1),
        "effort_reduction_pct": round(effort_reduction_pct, 1),
    },
    "estimates": rows,
    "clarifying_questions": clarifying,
}
OUT_PATH.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2),
                    encoding="utf-8")
print(f"[INFO] Agent-2 output -> {rel(OUT_PATH)}")

# Write Metric B back into process_graph.json so nb 03 (ROI) and the figure
# stage can read a single consistent set of counts.
G["counts"]["effort_reduction_pct"] = round(effort_reduction_pct, 1)
G["counts"]["asis_human_minutes_per_month"] = round(asis_human_min, 1)
G["counts"]["tobe_human_minutes_per_month"] = round(tobe_human_min, 1)
GRAPH_PATH.write_text(json.dumps(G, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[INFO] Metric B written back to {rel(GRAPH_PATH)}")

# Paper table: full per-node estimate table with source tags + reliability CV.
csv_path = TAB / "agent2_time_estimates.csv"
df2.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"[INFO] Paper table -> {rel(csv_path)}")

# Reliability summary: distribution of source tags + mean CV.
src_dist = df2["source"].value_counts().to_dict()
print(f"[INFO] Source-tag distribution: {src_dist}")
print(f"[INFO] Mean reliability CV (minutes): {df2['cv_minutes'].mean():.3f}")
print(f"[INFO] Agent-2 spend this run: ${COST.total_usd():.5f}")

[INFO] CostTracker, llm_call() re-established for nb 02.
[INFO] AS-IS nodes: 46  | interview: 5062 chars
[INFO] partial_retain (TO-BE residual human effort) = 0.3
[INFO] Shape priors (min): {'start': 3.0, 'task': 15.0, 'decision': 8.0, 'end': 5.0}
[INFO] Default cases/month: 20.0
[INFO] Self-Consistency: N=5 rollouts @ temp=0.7
[INFO] Agent-2 prompt ready (system 1730 chars, user 11214 chars).
[INFO] Running 5 Self-Consistency rollouts ...
   rollout 1/5: 46 estimates (cached=True, cost=$0.00252)
   rollout 2/5: 46 estimates (cached=True, cost=$0.00226)
   rollout 3/5: 46 estimates (cached=True, cost=$0.00244)
   rollout 4/5: 46 estimates (cached=True, cost=$0.00234)
   rollout 5/5: 46 estimates (cached=True, cost=$0.00219)
[INFO] Reduced 46 node estimates across 5 rollouts.
[INFO] AS-IS human effort : 41,050 min/month (684.2 h)
[INFO] TO-BE human effort : 19,755 min/month (329.2 h)
[Metric B] Human-effort reduction: 51.9%  <-- primary ROI input
[INFO] 3 clarifying question(s) generate